# 02F — Structural variation & interval biology


In [2]:
import pandas as pd
import numpy as np
import re
from pathlib import Path
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.stats import fisher_exact, chi2_contingency, mannwhitneyu, kruskal, spearmanr, ks_2samp, entropy, norm
from IPython.display import display

PROCESSED = Path('../../data/processed')
df = pd.read_csv(PROCESSED / 'DMD_variants_annotated.csv')

print('Loaded:', PROCESSED / 'DMD_variants_annotated.csv')
print('Shape:', df.shape)


Loaded: ../../data/processed/DMD_variants_annotated.csv
Shape: (11308, 30)


In [3]:
import sys
from pathlib import Path

PROJECT_ROOT = None
for rel in ('../..', '..', '.'):
    cand = (Path.cwd() / rel).resolve()
    if (cand / 'src' / 'utils.py').exists():
        PROJECT_ROOT = cand
        break
if PROJECT_ROOT is None:
    raise FileNotFoundError('project root with src/utils.py not found')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils import (
    add_consistency_flags,
    bh_adjust,
    chi2_table,
    ecdf_xy,
    fisher_bool,
    frame_mismatch,
    kruskal_group,
    mann_whitney_bool,
    mut_cons_mismatch,
    odds_ratio_ci,
    path_cons_mismatch,
    prepare_eda_dataframe,
    spearman_xy,
)

d = prepare_eda_dataframe(df)

print('Prepared shape:', d.shape)


Prepared shape: (11308, 59)


## 1. 📊 interval_length histogram


In [4]:
tmp = d[['interval_length_num']].dropna()
fig = px.histogram(tmp, x='interval_length_num', nbins=60, title='Interval length histogram')
fig.show()


## 2. 📊 log interval_length


In [5]:
tmp = d[['interval_length_num']].dropna().copy()
tmp['log_interval'] = np.log10(tmp['interval_length_num'])
fig = px.histogram(tmp, x='log_interval', nbins=60, title='log10 interval_length histogram')
fig.show()


## 3. 📊 interval_length vs phenotype


In [6]:
tmp = d[['phenotype', 'interval_length_num']].dropna()
fig = px.box(tmp, x='phenotype', y='interval_length_num', points='outliers', title='Interval length vs phenotype')
fig.update_yaxes(type='log')
fig.show()


## 4. 🧪 Kruskal interval_length ~ phenotype


In [7]:
names, groups, stat, p = kruskal_group(d, 'phenotype', 'interval_length_num')
display(pd.Series({'groups': ', '.join([str(x) for x in names]), 'kruskal_h': stat, 'p_value': p}).to_frame('value'))


,value
groups,"BMD, DMD"
kruskal_h,47.212415
p_value,0.0


## 5. 📊 interval_length vs pathogenic


In [8]:
tmp = d[['pathogenic', 'interval_length_num']].dropna()
fig = px.box(tmp, x='pathogenic', y='interval_length_num', points='outliers', title='Interval length vs pathogenic')
fig.update_yaxes(type='log')
fig.show()


## 6. 🧪 Mann–Whitney


In [9]:
g1, g0, stat, p = mann_whitney_bool(d, 'pathogenic', 'interval_length_num')
display(pd.Series({'n_pathogenic': len(g1), 'n_non_pathogenic': len(g0), 'u_statistic': stat, 'p_value': p}).to_frame('value'))


,value
n_pathogenic,1.938000e+03
n_non_pathogenic,5.790000e+02
u_statistic,7.158650e+05
p_value,3.561820e-24


## 7. 📊 interval_length vs exon


In [10]:
tmp = d[['exon_num', 'interval_length_num']].dropna().copy()
tmp['exon'] = tmp['exon_num'].astype(int)
fig = px.scatter(tmp, x='exon', y='interval_length_num', opacity=0.5, title='Interval length vs exon')
fig.update_yaxes(type='log')
fig.show()


## 8. 📊 interval_length vs domain


In [11]:
top = d['domain_clean'].value_counts().head(12).index
tmp = d[d['domain_clean'].isin(top)][['domain_clean', 'interval_length_num']].dropna()
fig = px.box(tmp, x='domain_clean', y='interval_length_num', points='outliers', title='Interval length vs domain (top)')
fig.update_xaxes(tickangle=-45)
fig.update_yaxes(type='log')
fig.show()


## 9. 🧪 Kruskal interval_length ~ domain


In [12]:
tmp = d[d['domain_clean'].isin(d['domain_clean'].value_counts()[lambda s: s >= 20].index)]
names, groups, stat, p = kruskal_group(tmp, 'domain_clean', 'interval_length_num')
display(pd.Series({'n_groups': len(names), 'kruskal_h': stat, 'p_value': p}).to_frame('value'))


,value
n_groups,29.000000
kruskal_h,29.299004
p_value,0.397482


## 10. 📊 hotspot vs interval_length ⭐


In [13]:
tmp = d[['is_hotspot_distal', 'interval_length_num']].dropna()
fig = px.box(tmp, x='is_hotspot_distal', y='interval_length_num', points='outliers', title='Distal hotspot vs interval_length')
fig.update_yaxes(type='log')
fig.show()


## 11. 🧪 Fisher large events in hotspot ⭐


In [14]:
cutoff = d['interval_length_num'].dropna().quantile(0.75)
tmp = d[['is_hotspot_distal', 'interval_length_num']].dropna().copy()
tmp['large_event'] = tmp['interval_length_num'] >= cutoff
tab, odds, p, ci = fisher_bool(tmp, 'is_hotspot_distal', 'large_event')
display(pd.Series({'large_event_cutoff': cutoff}).to_frame('value'))
display(tab)
display(pd.Series({'odds_ratio': odds, 'p_value': p, 'or_ci_low': ci[1], 'or_ci_high': ci[2]}).to_frame('value'))


,value
large_event_cutoff,276310.0


large_event,False,True
is_hotspot_distal,,
False,1429,481
True,457,150


,value
odds_ratio,0.975129
p_value,0.829969
or_ci_low,0.789315
or_ci_high,1.204686


## 12. 📊 ECDF interval_length DMD vs BMD


In [15]:
vals_dmd = d.loc[d['phenotype'] == 'DMD', 'interval_length_num'].dropna().values
vals_bmd = d.loc[d['phenotype'] == 'BMD', 'interval_length_num'].dropna().values
x1, y1 = ecdf_xy(vals_dmd)
x2, y2 = ecdf_xy(vals_bmd)
fig = go.Figure()
fig.add_trace(go.Scatter(x=x1, y=y1, mode='lines', name='DMD'))
fig.add_trace(go.Scatter(x=x2, y=y2, mode='lines', name='BMD'))
fig.update_layout(title='ECDF interval_length: DMD vs BMD', xaxis_title='interval_length', yaxis_title='ECDF')
fig.update_xaxes(type='log')
fig.show()


## 13. 🧪 KS test ⭐


In [16]:
vals_dmd = d.loc[d['phenotype'] == 'DMD', 'interval_length_num'].dropna()
vals_bmd = d.loc[d['phenotype'] == 'BMD', 'interval_length_num'].dropna()
stat, p = ks_2samp(vals_dmd, vals_bmd)
display(pd.Series({'n_dmd': len(vals_dmd), 'n_bmd': len(vals_bmd), 'ks_stat': stat, 'p_value': p}).to_frame('value'))


,value
n_dmd,1.558000e+03
n_bmd,1.040000e+02
ks_stat,4.265083e-01
p_value,1.403473e-16


## 14. 📊 interval_length vs frame_status


In [17]:
tmp = d[['frame', 'interval_length_num']].dropna()
fig = px.box(tmp, x='frame', y='interval_length_num', points='outliers', title='Interval length vs frame_status')
fig.update_yaxes(type='log')
fig.show()
